## Plan

#### 1. Preprocess and split

Use Lily’s code to:

- load breath-cycle data
- clean / encode / construct features
- split into train_data and test_data

#### 2. Add MFCC features

For each breath-cycle audio file:

- calculate MFCC features
- merge them onto train_data and test_data

Important: compute features separately on train and test files, don’t reshuffle after split

#### 3. Train + evaluate model

- Pass train_data and test_data into your model function.

#### ⭐️ TO DO ⭐️ 

- Check with Lily that disease encoder is fixed
- Get access to cycle file path and add to MFCC extraction

## 0. Imports

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
from scipy.stats import skew

sys.path.insert(0, os.path.abspath(".."))

from data_loading import DataLoader

from preprocessing import (
    ColumnEncoder,
    FeatureConstructor,
    get_breathing_cycle,
    Imputer,
    Scaler,
    stratified_group_split
)

from src.modeling import run_logistic_baseline

## 1. Preprocess and split tabular data

In [ ]:
# Load raw data
audio_path = "/Users/keira/code/mi-mi-mia/smart-stethoscope/raw_data/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files"
demographic_path = "/Users/keira/code/mi-mi-mia/smart-stethoscope/raw_data/demographic_info.txt"
diagnosis_path = "/Users/keira/code/mi-mi-mia/smart-stethoscope/raw_data/Respiratory_Sound_Database/Respiratory_Sound_Database/patient_diagnosis.csv"

loader = DataLoader(
    audio_path=audio_path,
    demographic_path=demographic_path,
    diagnosis_path=diagnosis_path
)

df = loader.transform(None)

In [ ]:
# Train/Test split at patient level
train_data, test_data = stratified_group_split(df)

In [ ]:
# Categorical Feature and Target Encoding
# TO DO: FIX DISEASE ENCODING
encoder = ColumnEncoder()
encoder.fit(train_data)

train_data = encoder.transform(train_data)
test_data = encoder.transform(test_data)

In [ ]:
# Feature Construction
constructor = FeatureConstructor()

train_data = constructor.transform(train_data)
test_data = constructor.transform(test_data)

In [ ]:
# Imputation (fit on train only)
imputer = Imputer()
imputer.fit(train_data)

train_data = imputer.transform(train_data)
test_data = imputer.transform(test_data)

In [ ]:
# Scaling (fit on train only)
scaler = Scaler()
scaler.fit(train_data)

train_data = scaler.transform(train_data)
test_data = scaler.transform(test_data)

## 2. Add MFCC Features 

In [ ]:
# MFCC feature extraction function
def extract_mfcc_features(df, audio_folder, n_mfcc=13):
    """
    Extract MFCC summary features for each filename in df.
    Assumes df has 'filename' column and corresponding .wav files exist.
    """
    mfcc_rows = []

    for filename in df["filename"]:
        file_path = Path(audio_folder) / f"{filename}.wav"

        signal, sample_rate = librosa.load(file_path, sr=None)

        mfcc = librosa.feature.mfcc(
            y=signal,
            sr=sample_rate,
            n_mfcc=n_mfcc
        )

        mfcc_mean = np.mean(mfcc, axis=1)
        mfcc_std = np.std(mfcc, axis=1)
        mfcc_skew = skew(mfcc, axis=1)
        mfcc_max = np.max(mfcc, axis=1)

        combined = np.concatenate([mfcc_mean, mfcc_std, mfcc_skew, mfcc_max])

        mfcc_rows.append([filename] + list(combined))

    columns = ["filename"]

    for i in range(1, n_mfcc + 1):
        columns.append(f"mfcc_{i}_mean")
    for i in range(1, n_mfcc + 1):
        columns.append(f"mfcc_{i}_std")
    for i in range(1, n_mfcc + 1):
        columns.append(f"mfcc_{i}_skew")
    for i in range(1, n_mfcc + 1):
        columns.append(f"mfcc_{i}_max")

    return pd.DataFrame(mfcc_rows, columns=columns)

In [ ]:
# Extract breathing cycles
sys.path.insert(0, os.path.abspath(".."))

preproc_audio_path = "/Users/keira/code/mi-mi-mia/smart-stethoscope/raw_data/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/processed_audio/"

annotation_data = extract_breathing_cycles(
    raw_audio_path=audio_path,
    preproc_audio_path=preproc_audio_path,
    max_len=6
)

In [ ]:
# Extract MFCCs separately for train/test
# IMPORTANT: use the folder containing the cycle .wav files
cycle_audio_path = preproc_audio_path

train_mfcc = extract_mfcc_features(train_data, cycle_audio_path)
test_mfcc = extract_mfcc_features(test_data, cycle_audio_path)

In [ ]:
# Merge MFCCs onto train/test
train_data = train_data.merge(train_mfcc, on="filename", how="left")
test_data = test_data.merge(test_mfcc, on="filename", how="left")

## 3. Train and evaluate model

In [ ]:
model, X_train, X_test, y_train, y_test, y_pred = run_logistic_baseline(
    train_data,
    test_data,
    target_col="disease",
    patient_col="pid"
)